# Module 5: AI/ML-Powered Data Quality

## Learning Objectives
- Detect anomalies in transaction data using statistical methods (z-score)
- Parse DQ rules from natural language using Cortex AI
- Run bulk AI table analysis with approve/reject governance workflow
- Auto-recommend system DMFs based on column types
- Build an end-to-end AI-assisted DQ pipeline

## Key Concept: AI-Augmented Quality

AI augments human DQ expertise in three ways:

| AI Capability | What It Does | When to Use |
|---------------|-------------|-------------|
| **Anomaly Detection** | Flags statistical outliers humans miss | Continuous numeric columns |
| **NL Rule Parsing** | Converts plain English to structured rule parameters | Quick rule creation by stewards |
| **Bulk Table Analysis** | Scans metadata and suggests multiple rules | Onboarding new tables |
| **System DMF Recommender** | Maps column types to appropriate system DMFs | Initial DQ setup |

The workflow: **AI suggests -> Human reviews -> Catalog stores -> Procedure provisions**

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~75 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## 5a. Statistical Anomaly Detection (Z-Score)

> **DQ Domain:** Validity (statistical outliers) | **Severity:** MEDIUM

Find transactions with amounts that are statistical outliers (> 2 standard deviations from the mean).

In [ ]:
WITH stats AS (
    SELECT AVG(AMOUNT) AS mean_amt, STDDEV(AMOUNT) AS std_amt
    FROM CORP_DWH.GOLD.FACT_TRANSACTIONS
),
scored AS (
    SELECT t.TXN_ID, t.CUSTOMER_ID, t.AMOUNT, t.TXN_TYPE, t.TXN_DATE,
        (t.AMOUNT - s.mean_amt) / NULLIF(s.std_amt, 0) AS z_score
    FROM CORP_DWH.GOLD.FACT_TRANSACTIONS t CROSS JOIN stats s
)
SELECT * FROM scored WHERE ABS(z_score) > 2 ORDER BY ABS(z_score) DESC LIMIT 10;

---
## 5b. Create Anomaly Detection DMF

> **DQ Domain:** Validity | **Severity:** MEDIUM

Codify the z-score logic as a reusable DMF that runs automatically.

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_AMOUNT_ANOMALIES(
    ARG_T TABLE(ARG_C NUMBER)
)
RETURNS NUMBER
AS
$$
    WITH stats AS (SELECT AVG(ARG_C) AS m, STDDEV(ARG_C) AS s FROM ARG_T)
    SELECT COUNT(*) FROM ARG_T CROSS JOIN stats
    WHERE ABS(ARG_C - m) / NULLIF(s, 0) > 2
$$;

ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_AMOUNT_ANOMALIES ON (AMOUNT);

---
## Checkpoint 1: Anomaly Detection

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

stats = session.sql("""
SELECT AVG(AMOUNT) AS MEAN_AMT, STDDEV(AMOUNT) AS STD_AMT, COUNT(*) AS TOTAL_ROWS
FROM CORP_DWH.GOLD.FACT_TRANSACTIONS
""").to_pandas()

anomalies = session.sql("""
WITH stats AS (SELECT AVG(AMOUNT) AS m, STDDEV(AMOUNT) AS s FROM CORP_DWH.GOLD.FACT_TRANSACTIONS)
SELECT COUNT(*) AS CNT FROM CORP_DWH.GOLD.FACT_TRANSACTIONS t CROSS JOIN stats
WHERE ABS(t.AMOUNT - m) / NULLIF(s, 0) > 2
""").collect()[0]['CNT']

print("=" * 50)
print("CHECKPOINT 1: Anomaly Detection")
print("=" * 50)
print(f"  Total transactions: {int(stats['TOTAL_ROWS'].iloc[0])}")
print(f"  Mean amount: {stats['MEAN_AMT'].iloc[0]:,.2f} SAR")
print(f"  Anomalies (|z| > 2): {anomalies}")
pct = 100 * anomalies / max(int(stats['TOTAL_ROWS'].iloc[0]), 1)
print(f"  Anomaly rate: {pct:.1f}%")
if anomalies > 0:
    print("  [PASS] Anomaly detection is working")
else:
    print("  [INFO] No anomalies found (possible with uniform data)")
print("=" * 50)

---
## 5c. AI-Assisted Rule Creation

> **Business Value:** Data stewards can create rules without SQL expertise. Time-to-rule drops from days (raise ticket, wait for engineer) to minutes (type in English, confirm).: Natural Language Parsing

> **DQ Domain:** All | **Severity:** Varies

This is the most powerful AI feature: **describe a rule in plain English** and let Cortex AI extract the structured parameters (rule type, regex, severity, target column).

This is exactly what data stewards use in the self-service Streamlit app -- they don't need to know SQL or regex syntax.

### How It Works
```
Steward types:  "National ID must be exactly 10 digits starting with 1 or 2"
                            |
                    Cortex AI parses
                            |
                            v
JSON output:    {"rule_name": "National ID Format",
                 "rule_type": "REGEX",
                 "target_column": "NATIONAL_ID", 
                 "regex_pattern": "^[12][0-9]{9}$",
                 "severity": "CRITICAL"}
```

In [ ]:
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# --- Try different natural language rule descriptions ---
# Change this text to test different rules!
nl_rule_description = "The email column must contain a valid email address with an @ symbol and a domain"

print(f"Input: \"{nl_rule_description}\"")
print("-" * 60)

# Build the AI prompt (same pattern as the Streamlit app)
prompt = (
    "You are a Data Quality rule parser for a Saudi enterprise data warehouse. "
    "Given a natural language rule description, extract structured parameters. "
    "Return ONLY valid JSON with these fields: "
    "rule_name (string), rule_type (one of: REGEX, RANGE, ENUM, COMPLETENESS, UNIQUENESS), "
    "target_column (string, UPPERCASE), regex_pattern (string or null), "
    "enum_values (comma-separated string or null), "
    "min_value (number or null), max_value (number or null), "
    "severity (one of: LOW, MEDIUM, HIGH, CRITICAL), "
    "description (one sentence explaining the rule). "
    f"\nRule description: {nl_rule_description}"
)

# Call Cortex AI
result = session.sql(
    f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', $${prompt}$$) AS RESPONSE"
).collect()[0]['RESPONSE']

# Parse the JSON from AI response
try:
    # AI sometimes wraps JSON in markdown code blocks -- strip them
    clean = result.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1]  # remove first line
        clean = clean.rsplit("```", 1)[0]  # remove last ```
    parsed = json.loads(clean)
    
    print("\nAI Parsed Rule:")
    print("=" * 60)
    print(f"  Rule Name:     {parsed.get('rule_name', 'N/A')}")
    print(f"  Rule Type:     {parsed.get('rule_type', 'N/A')}")
    print(f"  Target Column: {parsed.get('target_column', 'N/A')}")
    print(f"  Severity:      {parsed.get('severity', 'N/A')}")
    if parsed.get('regex_pattern'):
        print(f"  Regex Pattern: {parsed['regex_pattern']}")
    if parsed.get('enum_values'):
        print(f"  Enum Values:   {parsed['enum_values']}")
    if parsed.get('min_value') is not None:
        print(f"  Min Value:     {parsed['min_value']}")
    if parsed.get('max_value') is not None:
        print(f"  Max Value:     {parsed['max_value']}")
    print(f"  Description:   {parsed.get('description', 'N/A')}")
    print("=" * 60)
    print("\n[PASS] AI successfully parsed the natural language rule!")
    
except json.JSONDecodeError as e:
    print(f"\n[WARN] AI returned non-JSON response. Raw output:")
    print(result[:500])
    parsed = None

### Try It Yourself!

Change the `nl_rule_description` variable above to test different rules:

| Try This | Expected Output |
|----------|----------------|
| "Phone must start with +966 or 05" | REGEX, `^(\+966|05)[0-9]{8,9}$` |
| "Transaction amount must be between 0 and 1 million SAR" | RANGE, min=0, max=1000000 |
| "City must be one of Riyadh, Jeddah, Dammam, Makkah" | ENUM, values="Riyadh,Jeddah,..." |
| "Customer name cannot be empty" | COMPLETENESS |
| "National ID must be unique across all records" | UNIQUENESS |

---
## 5d. Bulk AI Table Analysis

> **Business Value:** Onboarding a new data source typically takes weeks of analysis. AI suggests rules in seconds, reducing onboarding from weeks to hours. with Approve/Reject Workflow

> **DQ Domain:** All | **Severity:** Varies

Instead of describing one rule at a time, AI can analyze an **entire table** and suggest multiple rules at once. These are inserted into the catalog as `OWNER = 'AI_SUGGESTED'` with `IS_ACTIVE = FALSE` -- they require human approval before being provisioned.

This is the governance workflow:
```
AI generates suggestions --> Inserted as PENDING --> Human reviews --> Approve/Reject --> Provision
```

In [ ]:
-- Create the AI suggestion procedure
CREATE OR REPLACE PROCEDURE CORP_DWH.DQ.AI_SUGGEST_RULES(TABLE_PATH STRING)
RETURNS STRING
LANGUAGE SQL
COMMENT = 'Analyzes a table with Cortex AI and inserts suggested rules into the catalog'
AS
DECLARE
    col_info STRING;
    prompt STRING;
    ai_response STRING;
    clean_response STRING;
    rules_added NUMBER DEFAULT 0;
BEGIN
    -- Get column metadata
    LET parts ARRAY := SPLIT(:TABLE_PATH, '.');
    LET tbl_schema STRING := parts[0];
    LET tbl_name STRING := parts[1];
    
    SELECT LISTAGG(COLUMN_NAME || ' (' || DATA_TYPE || ')', ', ')
    INTO :col_info
    FROM CORP_DWH.INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = :tbl_schema AND TABLE_NAME = :tbl_name;

    -- Build prompt
    prompt := 'You are a Data Quality expert for a Saudi enterprise. Suggest exactly 3 DQ rules for table CORP_DWH.' ||
        :TABLE_PATH || '. Columns: ' || :col_info ||
        '. Return ONLY a valid JSON array (no markdown, no explanation) with objects containing: ' ||
        'rule_name (string), rule_type (REGEX/RANGE/ENUM), ' ||
        'target_column (UPPERCASE), regex_pattern (string or null), enum_values (comma-separated string or null), ' ||
        'min_value (number or null), max_value (number or null), severity (LOW/MEDIUM/HIGH/CRITICAL), description (string). ' ||
        'Focus on Saudi-specific business validations.';

    -- Call Cortex AI
    SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', :prompt) INTO :ai_response;
    
    -- Strip markdown code fences if present (AI sometimes wraps in ```json...```)
    clean_response := REGEXP_REPLACE(:ai_response, '^[\\s]*```[a-z]*\\n?', '');
    clean_response := REGEXP_REPLACE(:clean_response, '\\n?```[\\s]*$', '');
    clean_response := TRIM(:clean_response);

    -- Parse JSON array and insert each rule into catalog
    INSERT INTO CORP_DWH.DQ.RULES_CATALOG
        (RULE_NAME, RULE_TYPE, TARGET_SCHEMA, TARGET_TABLE, TARGET_COLUMN,
         REGEX_PATTERN, MIN_VALUE, MAX_VALUE, ENUM_VALUES, SEVERITY, OWNER, IS_ACTIVE)
    SELECT
        f.value:rule_name::STRING,
        f.value:rule_type::STRING,
        :tbl_schema,
        :tbl_name,
        f.value:target_column::STRING,
        f.value:regex_pattern::STRING,
        f.value:min_value::NUMBER,
        f.value:max_value::NUMBER,
        f.value:enum_values::STRING,
        COALESCE(f.value:severity::STRING, 'MEDIUM'),
        'AI_SUGGESTED',
        FALSE
    FROM TABLE(FLATTEN(TRY_PARSE_JSON(:clean_response))) f
    WHERE f.value:rule_name IS NOT NULL;

    rules_added := SQLROWCOUNT;
    
    -- Fallback: if JSON parsing failed, insert a placeholder so student knows AI responded
    IF (:rules_added = 0) THEN
        INSERT INTO CORP_DWH.DQ.RULES_CATALOG
            (RULE_NAME, RULE_TYPE, TARGET_SCHEMA, TARGET_TABLE, TARGET_COLUMN, SEVERITY, OWNER, IS_ACTIVE)
        VALUES
            ('AI_PARSE_FAILED_' || :tbl_name, 'REGEX', :tbl_schema, :tbl_name, 'REVIEW_AI_OUTPUT',
             'MEDIUM', 'AI_SUGGESTED', FALSE);
        rules_added := 1;
    END IF;

    RETURN 'AI analyzed ' || :TABLE_PATH || '. Inserted ' || :rules_added || ' suggestion(s). Review: SELECT * FROM CORP_DWH.DQ.RULES_CATALOG WHERE OWNER = ''AI_SUGGESTED''';
END;

### Run the AI Analyzer

> **What this does:** Executes the AI_SUGGEST_RULES procedure and shows the result.

In [ ]:
-- Analyze FACT_TRANSACTIONS table
CALL CORP_DWH.DQ.AI_SUGGEST_RULES('GOLD.FACT_TRANSACTIONS');

### Review Pending Suggestions

> **What this does:** Queries the rules catalog to show all AI-suggested rules that are awaiting human approval.

In [ ]:
-- View all AI-suggested rules awaiting approval
SELECT RULE_ID, RULE_NAME, RULE_TYPE, TARGET_TABLE, TARGET_COLUMN, SEVERITY, IS_ACTIVE
FROM CORP_DWH.DQ.RULES_CATALOG
WHERE OWNER = 'AI_SUGGESTED'
ORDER BY CREATED_AT DESC;

### Approve or Reject

In the Streamlit app, stewards click Approve/Reject buttons. In the notebook, we use SQL:

In [ ]:
-- APPROVE a suggestion (set IS_ACTIVE = TRUE)
UPDATE CORP_DWH.DQ.RULES_CATALOG
SET IS_ACTIVE = TRUE
WHERE OWNER = 'AI_SUGGESTED' AND IS_ACTIVE = FALSE;

-- To REJECT instead (delete):
-- DELETE FROM CORP_DWH.DQ.RULES_CATALOG WHERE RULE_ID = <id>;

---
## 5e. System DMF Auto-Recommender

> **Business Value:** Ensures no table goes unmonitored. Automated recommendation eliminates the 'forgot to add checks' gap that causes silent data degradation.

> **DQ Domain:** All | **Severity:** Varies

This feature analyzes column data types and recommends which **system DMFs** to attach. No AI/LLM needed -- pure logic based on column types:

| Column Type | Recommended DMFs |
|------------|-----------------|
| STRING/VARCHAR | NULL_COUNT, BLANK_COUNT, DUPLICATE_COUNT |
| TIMESTAMP | FRESHNESS, NULL_COUNT |
| NUMBER | NULL_COUNT |
| Table-level | ROW_COUNT |

The Streamlit app does this automatically. Here we build it as a Python function.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

def recommend_system_dmfs(schema: str, table: str) -> list:
    """Analyze column types and recommend system DMFs to attach."""
    cols = session.sql(f"""
        SELECT COLUMN_NAME, DATA_TYPE
        FROM CORP_DWH.INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{table}'
        ORDER BY ORDINAL_POSITION
    """).to_pandas()
    
    # Check what's already attached
    try:
        attached = session.sql(f"""
            SELECT METRIC_NAME, ARGUMENTS
            FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
                REF_ENTITY_NAME => 'CORP_DWH.{schema}.{table}', REF_ENTITY_DOMAIN => 'TABLE'))
        """).to_pandas()
        attached_set = set(attached['METRIC_NAME'].tolist()) if not attached.empty else set()
    except:
        attached_set = set()
    
    recommendations = []
    
    # Table-level: always recommend ROW_COUNT
    if 'ROW_COUNT' not in attached_set:
        recommendations.append({
            'dmf': 'SNOWFLAKE.CORE.ROW_COUNT',
            'column': '()',
            'reason': 'Volume check - confirm data arrives'
        })
    
    for _, col in cols.iterrows():
        col_name = col['COLUMN_NAME']
        dtype = col['DATA_TYPE'].upper()
        
        if 'VARCHAR' in dtype or 'TEXT' in dtype or 'STRING' in dtype:
            recommendations.append({'dmf': 'SNOWFLAKE.CORE.NULL_COUNT', 'column': col_name, 'reason': 'Completeness'})
            recommendations.append({'dmf': 'SNOWFLAKE.CORE.BLANK_COUNT', 'column': col_name, 'reason': 'Empty strings'})
        
        elif 'TIMESTAMP' in dtype:
            recommendations.append({'dmf': 'SNOWFLAKE.CORE.FRESHNESS', 'column': col_name, 'reason': 'SLA monitoring'})
            recommendations.append({'dmf': 'SNOWFLAKE.CORE.NULL_COUNT', 'column': col_name, 'reason': 'Completeness'})
        
        elif 'NUMBER' in dtype or 'INT' in dtype or 'FLOAT' in dtype:
            recommendations.append({'dmf': 'SNOWFLAKE.CORE.NULL_COUNT', 'column': col_name, 'reason': 'Completeness'})
    
    return recommendations

# Run recommender on a table we haven't monitored much
print("System DMF Recommendations for SILVER.INT_TRANSACTIONS")
print("=" * 70)
recs = recommend_system_dmfs('SILVER', 'INT_TRANSACTIONS')

print(f"\n  Found {len(recs)} recommendations:\n")
print(f"  {'DMF':<35} {'Column':<20} {'Reason'}")
print(f"  {'-'*35} {'-'*20} {'-'*20}")
for r in recs[:12]:  # limit display
    print(f"  {r['dmf']:<35} {r['column']:<20} {r['reason']}")

print(f"\n  Total: {len(recs)} DMFs recommended")
print("=" * 70)

### Generate and Execute the ALTER TABLE Statements

> **What this does:** Takes the DMF recommendations from above and generates + executes the ALTER TABLE statements to attach them to your tables.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Generate ALTER TABLE statements from recommendations
# (Using a subset to avoid attaching too many DMFs in a lab)
target_schema = 'SILVER'
target_table = 'INT_TRANSACTIONS'

key_columns = {
    'FRESHNESS': ['LOADED_AT'],
    'NULL_COUNT': ['CUSTOMER_ID', 'TXN_TYPE'],
    'BLANK_COUNT': ['TXN_TYPE'],
}

print("Attaching recommended system DMFs...")
print("=" * 50)
attached = 0

for dmf_short, columns in key_columns.items():
    for col in columns:
        try:
            if dmf_short == 'FRESHNESS':
                stmt = f"ALTER TABLE CORP_DWH.{target_schema}.{target_table} ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.{dmf_short} ON ({col})"
            else:
                stmt = f"ALTER TABLE CORP_DWH.{target_schema}.{target_table} ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.{dmf_short} ON ({col})"
            session.sql(stmt).collect()
            print(f"  [OK] {dmf_short} on {col}")
            attached += 1
        except Exception as e:
            if 'already exists' in str(e).lower() or 'already associated' in str(e).lower():
                print(f"  [SKIP] {dmf_short} on {col} (already attached)")
            else:
                print(f"  [ERR] {dmf_short} on {col}: {str(e)[:60]}")

print(f"\n  Attached {attached} new DMFs")
print("=" * 50)

---
## 5f. End-to-End: AI-Assisted DQ Pipeline (Capstone)

> **DQ Domain:** All | **Severity:** All

Put it all together: use AI to fully onboard a table for DQ monitoring in one workflow.

**Target:** `CORP_DWH.GOLD.FACT_TRANSACTIONS` (we've only attached the anomaly DMF so far)

Steps:
1. System DMF recommender (auto-attach volume, freshness, null checks)
2. Cortex AI suggests business rules
3. Insert AI suggestions into catalog
4. Provision everything

In [ ]:
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("END-TO-END AI-ASSISTED DQ PIPELINE")
print("Target: CORP_DWH.GOLD.FACT_TRANSACTIONS")
print("=" * 60)

# Step 1: Attach key system DMFs
print("\n--- Step 1: System DMF Attachment ---")
system_dmfs = [
    ("SNOWFLAKE.CORE.ROW_COUNT", "()"),
    ("SNOWFLAKE.CORE.FRESHNESS", "(LOADED_AT)"),
    ("SNOWFLAKE.CORE.NULL_COUNT", "(CUSTOMER_ID)"),
]

for dmf, col_expr in system_dmfs:
    try:
        session.sql(f"ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS ADD DATA METRIC FUNCTION {dmf} ON {col_expr}").collect()
        print(f"  [OK] {dmf} ON {col_expr}")
    except Exception as e:
        if 'already' in str(e).lower():
            print(f"  [SKIP] {dmf} ON {col_expr} (already attached)")
        else:
            print(f"  [ERR] {str(e)[:60]}")

# Step 2: AI suggests a business rule via natural language
print("\n--- Step 2: AI Rule from Natural Language ---")
nl_input = "Transaction amounts should never exceed 100,000 SAR for a single transaction"

prompt = (
    "You are a DQ rule parser. Extract parameters from this rule: "
    f"\"{nl_input}\" "
    "Return ONLY valid JSON: {rule_name, rule_type, target_column, "
    "regex_pattern, enum_values, min_value, max_value, severity, description}"
)

ai_result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', $${prompt}$$) AS R").collect()[0]['R']

try:
    clean = ai_result.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    parsed = json.loads(clean)
    print(f"  AI parsed: {parsed.get('rule_name', 'N/A')} ({parsed.get('rule_type', 'N/A')})")
    print(f"  Max value: {parsed.get('max_value', 'N/A')}, Severity: {parsed.get('severity', 'N/A')}")
except:
    print(f"  [WARN] Could not parse AI response, using fallback")
    parsed = {"rule_name": "Max Transaction Amount", "rule_type": "RANGE", 
              "target_column": "AMOUNT", "min_value": 0, "max_value": 100000, "severity": "HIGH"}

# Step 3: Insert into catalog
print("\n--- Step 3: Insert into Rules Catalog ---")
rule_name = parsed.get('rule_name', 'Max Transaction Amount')
rule_type = parsed.get('rule_type', 'RANGE')
target_col = parsed.get('target_column', 'AMOUNT')
min_val = parsed.get('min_value', 0) or 0
max_val = parsed.get('max_value', 100000) or 100000
severity = parsed.get('severity', 'HIGH')

try:
    session.sql(f"""
        INSERT INTO CORP_DWH.DQ.RULES_CATALOG
            (RULE_NAME, RULE_TYPE, TARGET_SCHEMA, TARGET_TABLE, TARGET_COLUMN,
             MIN_VALUE, MAX_VALUE, SEVERITY, OWNER, IS_ACTIVE)
        VALUES
            ('{rule_name}', '{rule_type}', 'GOLD', 'FACT_TRANSACTIONS', '{target_col}',
             {min_val}, {max_val}, '{severity}', 'AI_SUGGESTED', TRUE)
    """).collect()
    print(f"  [OK] Inserted: {rule_name} (auto-approved for demo)")
except Exception as e:
    print(f"  [INFO] {str(e)[:60]}")

# Step 4: Provision
print("\n--- Step 4: Provision DMFs from Catalog ---")
try:
    result = session.sql("CALL CORP_DWH.DQ.PROVISION_DMFS_FROM_CATALOG()").collect()[0][0]
    print(f"  [OK] {result}")
except Exception as e:
    print(f"  [INFO] {str(e)[:60]}")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE!")
print("FACT_TRANSACTIONS now has: system DMFs + anomaly DMF + AI-suggested business rule")
print("=" * 60)

---
## Final Checkpoint: AI Features Verification

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("FINAL CHECKPOINT: AI/ML Module Verification")
print("=" * 60)
passed = 0
total = 5

# 1. Anomaly DMF exists
try:
    session.sql("DESCRIBE FUNCTION CORP_DWH.DQ.CHECK_AMOUNT_ANOMALIES(TABLE(NUMBER))").collect()
    print("  [PASS] 1/5 - CHECK_AMOUNT_ANOMALIES DMF exists")
    passed += 1
except:
    print("  [FAIL] 1/5 - CHECK_AMOUNT_ANOMALIES DMF not found")

# 2. Cortex AI is accessible
try:
    session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', 'Say OK') AS R").collect()
    print("  [PASS] 2/5 - Cortex AI is accessible")
    passed += 1
except:
    print("  [FAIL] 2/5 - Cortex AI not available")

# 3. AI_SUGGEST_RULES procedure exists
try:
    session.sql("DESCRIBE PROCEDURE CORP_DWH.DQ.AI_SUGGEST_RULES(STRING)").collect()
    print("  [PASS] 3/5 - AI_SUGGEST_RULES procedure exists")
    passed += 1
except:
    print("  [FAIL] 3/5 - AI_SUGGEST_RULES procedure not found")

# 4. FACT_TRANSACTIONS has multiple DMFs attached
try:
    dmf_count = session.sql("""
        SELECT COUNT(*) AS CNT FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
            REF_ENTITY_NAME => 'CORP_DWH.GOLD.FACT_TRANSACTIONS', REF_ENTITY_DOMAIN => 'TABLE'))
    """).collect()[0]['CNT']
    if dmf_count >= 2:
        print(f"  [PASS] 4/5 - FACT_TRANSACTIONS has {dmf_count} DMFs attached")
        passed += 1
    else:
        print(f"  [FAIL] 4/5 - Only {dmf_count} DMFs on FACT_TRANSACTIONS (expected >= 2)")
except:
    print("  [FAIL] 4/5 - Could not check DMF references")

# 5. AI-suggested rules exist in catalog
ai_rules = session.sql("""
    SELECT COUNT(*) AS CNT FROM CORP_DWH.DQ.RULES_CATALOG WHERE OWNER = 'AI_SUGGESTED'
""").collect()[0]['CNT']
if ai_rules > 0:
    print(f"  [PASS] 5/5 - {ai_rules} AI-suggested rules in catalog")
    passed += 1
else:
    print("  [FAIL] 5/5 - No AI-suggested rules found")

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 60)

---
## Quiz: Test Your Knowledge

**Q1:** In the NL rule parsing workflow, why do we require human review before inserting into the catalog? Why not auto-insert?

**Q2:** The system DMF recommender maps STRING columns to NULL_COUNT + BLANK_COUNT. Why both? Give a real scenario where one catches an issue the other misses.

**Q3:** The bulk AI analysis inserts rules with `IS_ACTIVE = FALSE`. What happens if you run `PROVISION_DMFS_FROM_CATALOG()` before approving them?

**Q4:** You're onboarding 50 new tables. Rank these AI features by time saved: (a) NL parsing, (b) bulk analysis, (c) system DMF recommender.

**Q5:** Can Cortex AI replace the Rules Catalog entirely? Why or why not?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: Human review is essential because:
    - AI may hallucinate invalid regex patterns
    - AI doesn't know business context (e.g., "negative amounts are valid for refunds")
    - AI may suggest overly strict rules that create false positives
    - Compliance requires human accountability for DQ decisions
    The pattern is: AI SUGGESTS, human APPROVES, system PROVISIONS.

Q2: Real scenario:
    - NULL_COUNT catches: WHERE NATIONAL_ID IS NULL (field absent from source)
    - BLANK_COUNT catches: WHERE NATIONAL_ID = '' (field present but empty string)
    CRM systems often send empty strings instead of NULL when a field is "not filled in."
    If you only check NULL_COUNT, you'd miss 100% of these phantom records.
    Both together = complete "is this field actually populated?" coverage.

Q3: Nothing happens to inactive rules! The procedure checks:
    WHERE IS_ACTIVE = TRUE AND DMF_NAME IS NULL
    So IS_ACTIVE = FALSE rules are skipped entirely. They sit in the catalog
    as pending suggestions until approved. This is the governance safeguard.

Q4: Time saved ranking for 50 tables:
    1. (c) System DMF recommender - HIGHEST. Purely automated, no AI latency,
       attaches 5-10 DMFs per table instantly. 50 tables x 8 DMFs = 400 checks in seconds.
    2. (b) Bulk analysis - MEDIUM. One AI call per table, but requires review.
       50 calls x 3 rules = 150 suggestions to review.
    3. (a) NL parsing - LOWEST for bulk. It's one rule at a time.
       Best for ad-hoc additions, not bulk onboarding.

Q5: No. Cortex AI cannot replace the Rules Catalog because:
    - AI is non-deterministic (different suggestions each run)
    - Rules need versioning, ownership, and audit trails
    - Provisioned DMFs must map to a catalog record for governance
    - AI doesn't know when rules should be disabled or updated
    - Regulatory compliance requires documented, traceable rule definitions
    AI is a DISCOVERY tool; the Catalog is the SYSTEM OF RECORD.
""")

---
## Summary

| Section | AI Technique | DQ Domain |
|---------|-------------|-----------|
| 5a-5b | Z-score anomaly detection | Validity |
| 5c | Natural language rule parsing (Cortex COMPLETE) | All |
| 5d | Bulk table analysis + approve/reject | All |
| 5e | System DMF recommender (type-based logic) | Volume, Freshness, Completeness |
| 5f | End-to-end AI pipeline (capstone) | All |

**Key Insight:** AI is most powerful as a **discovery accelerator** -- it finds rules humans haven't thought of. But it must always feed into a governed catalog with human oversight.

---

**Next:** Open `6_GOVERNANCE` to connect DQ with Snowflake Horizon.